# Tutorial 0: Camera Calibration with Anipose

**Pipeline Stage:** Producing the `calibration.toml` that 3D triangulation depends on

---

## Why This Tutorial Exists

Tutorials 1–3 take you from raw multi-camera video all the way to 3D skeletons.
But **Tutorial 3 (3D Triangulation) assumes you already have a `calibration.toml`**
for your camera rig — and never shows you how to make one.

This tutorial fills that gap. It takes you from **raw calibration videos** (one per
camera, all recording the same moving calibration board) to a finished
**`calibration.toml`**, using [Anipose](https://github.com/lambdaloop/anipose) /
[`sleap-anipose`](https://github.com/talmolab/sleap-anipose).

```
┌──────────────────────┐     ┌──────────────────────┐     ┌──────────────────────┐
│  Tutorial 0 (HERE)   │ ──► │  Tutorials 1 & 2     │ ──► │  Tutorial 3          │
│  Camera Calibration  │     │  2D Pose + ReID       │     │  3D Triangulation    │
│  → calibration.toml  │     │                       │     │  (uses calibration)  │
└──────────────────────┘     └──────────────────────┘     └──────────────────────┘
```

## What Calibration Actually Solves

To triangulate a 2D point seen in several cameras into a single 3D point, you must
know, for every camera:

1. **Intrinsics** — focal length, principal point, and lens distortion. *How does a
   3D point in front of this camera land on a pixel?*
2. **Extrinsics** — rotation and translation. *Where is this camera sitting in the
   world, and which way is it pointing?*

Calibration recovers all of these at once by watching a **known object** — a
calibration board with a precisely known geometry — move through the shared field of
view of every camera. Anipose detects the board corners in each view and runs
**iterative bundle adjustment** to jointly solve for every camera's intrinsics and
extrinsics while minimizing **reprojection error** (the pixel gap between where a
corner *was* detected and where the solved model says it *should* be).

## What You Will Learn

1. **Installation** — git clone → conda env → `sleap-anipose`, end to end
2. **The calibration board** — what a ChArUco board is, and printing your own
3. **Recording a good calibration video** — the single biggest driver of quality
4. **The session folder format** — camera subfolders + `calibration_images/`
5. **Building synchronized calibration videos** — raw video → one shared frame list →
   `calibration_images/*-calibration.mp4`, keeping cameras index-aligned
6. **Running calibration** — one `slap.calibrate(...)` call → `calibration.toml`
7. **Reading `calibration.toml`** — what every field means
8. **Judging quality** — reprojection-error histogram + overlays, and what's "good"

---

## Part 0: Installation (from scratch)

Calibration uses `sleap-anipose`, which wraps the `aniposelib` calibration engine.
Below is a complete setup starting from nothing. You only need to do this **once**.

> If you already made the `multicam-pose` environment from the main `README.md`, you
> can reuse it — just make sure `sleap-anipose` is installed in it (see step 4).

### 1. Install a conda/mamba distribution (skip if you already have one)

If you don't have `conda`, install [Miniforge](https://github.com/conda-forge/miniforge)
(recommended — ships the fast `mamba` solver and the conda-forge channel by default).

### 2. Clone the repositories

```bash
# The tutorials repo (this folder lives inside it) and sleap-anipose for reference
git clone https://github.com/talmolab/sleap-anipose.git
git clone https://github.com/lambdaloop/anipose.git   # optional: docs & reference
```

You do **not** need to install from the clone — `sleap-anipose` is on PyPI — but the
clone gives you `docs/FOLDER_STRUCTURE.md` and the source to read.

### 3. Create and activate the conda environment

```bash
conda create -n sleap-anipose python=3.9 -y
conda activate sleap-anipose
```

### 4. Install the calibration stack

```bash
# Core calibration engine (pulls in aniposelib, opencv, numba, toml, imageio, etc.)
pip install sleap-anipose

# Notebook + plotting utilities used in this tutorial
pip install jupyter ipykernel matplotlib pandas
```

### 5. Register a Jupyter kernel so this notebook can find the env

```bash
python -m ipykernel install --user \
    --name sleap-anipose \
    --display-name "Python (sleap-anipose)"
```

Then, in Jupyter, pick the **"Python (sleap-anipose)"** kernel (top-right) before
running the cells below.

> **OpenCV / ArUco note:** `sleap-anipose` depends on `opencv-contrib` for the ArUco
> module. `pip install sleap-anipose` handles this. If you ever see
> `module 'cv2' has no attribute 'aruco'`, you have a plain `opencv-python` shadowing
> it — fix with:
> `pip uninstall -y opencv-python opencv-contrib-python && pip install opencv-contrib-python`.

In [ ]:
# ============================================================
# STEP 0: Verify the environment
# ============================================================
import sys, platform
print(f"Python:   {sys.version.split()[0]}  ({platform.system()})")

import cv2
print(f"OpenCV:   {cv2.__version__}")
assert hasattr(cv2, "aruco"), (
    "cv2.aruco missing — install opencv-contrib-python (see the note above)."
)

import numpy as np
import matplotlib
print(f"numpy:    {np.__version__}")
print(f"mpl:      {matplotlib.__version__}")

try:
    import sleap_anipose as slap
    import aniposelib
    print(f"sleap-anipose: OK  |  aniposelib: {getattr(aniposelib, '__version__', 'installed')}")
except Exception as e:
    print(f"WARNING: could not import sleap_anipose ({e}).")
    print("Re-check Part 0 and that you selected the 'Python (sleap-anipose)' kernel.")

---

## Part 1: The Calibration Board

Anipose supports **checkerboards**, **ArUco** boards, and **ChArUco** boards.
`sleap-anipose` standardizes on the **ChArUco** board, and so will we.

### Why ChArUco?

A ChArUco board is a chessboard with an ArUco marker inside every white square:

```
┌───┬───┬───┬───┐
│▪ ▪│███│▪ ▪│███│   ███  = black chessboard square
├───┼───┼───┼───┤   ▪ ▪  = ArUco marker (a unique binary tag)
│███│▪ ▪│███│▪ ▪│
├───┼───┼───┼───┤   • Chessboard corners → sub-pixel accurate positions
│▪ ▪│███│▪ ▪│███│   • ArUco tags         → identify WHICH corner is which,
└───┴───┴───┴───┘                          even when the board is partly cut off
```

The chessboard gives **precision**; the ArUco tags give **unique identity** for each
corner. That combination means the board still calibrates correctly even when it's
tilted, partly out of frame, or only partly overlapping between two cameras — which is
exactly what happens when you wave it around a multi-camera volume.

### Board parameters

A ChArUco board is fully described by six numbers. These **must match your physical
board** — most importantly the lengths, which set the real-world **units** of your
entire 3D reconstruction.

| Parameter | Meaning | Example |
|---|---|---|
| `board_x` | squares across the width | `8` |
| `board_y` | squares down the height | `11` |
| `square_length` | chessboard square edge, **in your chosen units** | `24.0` (mm) |
| `marker_length` | ArUco marker edge, same units | `18.75` (mm) |
| `marker_bits` | bits per ArUco marker (4/5/6/7) | `4` |
| `dict_size` | ArUco dictionary size (50/100/250/1000) | `1000` |

> **Units set the world scale.** If `square_length` is in millimetres, every 3D
> coordinate you triangulate later (Tutorial 3) will be in millimetres. Measure your
> printed board's square edge with calipers and put the *real* number here.

In [ ]:
# ============================================================
# STEP 1: Define the calibration board
# ============================================================
# EDIT THESE to match YOUR physical board.
BOARD = {
    "board_x": 8,            # squares across (width)
    "board_y": 11,           # squares down (height)
    "square_length": 24.0,   # square edge length -> sets world units (mm here)
    "marker_length": 18.75,  # ArUco marker edge length (same units)
    "marker_bits": 4,        # 4x4 markers
    "dict_size": 1000,       # DICT_4X4_1000
}

# Persist the board spec next to your data so calibration is reproducible.
# This writes a board.toml that slap.calibrate() can also read directly.
import os
PROJECT_DIR = "calibration_demo"            # working directory for this tutorial
os.makedirs(PROJECT_DIR, exist_ok=True)
board_toml = os.path.join(PROJECT_DIR, "board.toml")

slap.write_board(board_name=board_toml, **BOARD)
print(f"Wrote board spec -> {board_toml}")
print(open(board_toml).read())

### Print your own board

You need a **physical** copy of the board to record calibration video. `sleap-anipose`
can draw a printable one for you. Print it at 100% scale (no "fit to page"), mount it
on something **rigid and flat** (foam board, clipboard), then **re-measure** the actual
printed square size and update `square_length` / `marker_length` above if it drifted.

In [ ]:
# ============================================================
# STEP 2: Draw a printable ChArUco board
# ============================================================
board_png = os.path.join(PROJECT_DIR, "charuco_board.png")

slap.draw_board(
    board_name=board_png,
    board_x=BOARD["board_x"],
    board_y=BOARD["board_y"],
    square_length=BOARD["square_length"],
    marker_length=BOARD["marker_length"],
    marker_bits=BOARD["marker_bits"],
    dict_size=BOARD["dict_size"],
    img_width=1440,
    img_height=1980,      # ~ width * board_y / board_x keeps squares square
    save="",              # (optional) path to also dump a board.toml
)

import matplotlib.pyplot as plt
img = plt.imread(board_png)
plt.figure(figsize=(6, 8))
plt.imshow(img, cmap="gray")
plt.title("Printable ChArUco board\n(print at 100% scale, mount flat & rigid)")
plt.axis("off")
plt.show()
print(f"Saved printable board -> {board_png}")

---

## Part 2: Recording a Good Calibration Video

**This is the step that decides your calibration quality.** The math is only as good as
the board coverage you feed it. Record one synchronized clip per camera of the board
being moved slowly through the capture volume.

### The rules that matter

- **Every camera must see the board a lot.** Bundle adjustment can only relate two
  cameras through frames where *both* see the board. Move so that overlapping pairs get
  many shared views.
- **Fill the whole volume.** Walk the board through the entire 3D space your subjects
  will occupy — near/far, left/right, floor/height. Corners of the volume matter most.
- **Tilt and rotate the board.** Vary its angle (pitch/yaw/roll), not just its
  position. Head-on-only views make focal length and distortion poorly constrained.
- **Move slowly / pause.** Rolling-shutter motion blur ruins corner detection. Glide,
  and hold briefly at each pose. Good lighting, no glare on the board.
- **Keep it flat and rigid.** Any bend in the board violates the known geometry and
  poisons the solve.
- **Enough frames.** After keeping only frames where the board is clearly visible, aim
  for **~100–300 good frames per camera**. A 1–3 minute clip usually gets you there.

### Synchronization

The cameras don't need microsecond sync for calibration (the board is roughly static
during each slow pose), but roughly aligned clips help. If your rig hardware-syncs for
the real recordings, just reuse that.

```
        Move the board through the WHOLE volume, tilting as you go:

           near ─────────────────────────► far
            ┌───────────────────────────────┐
            │   ◹      ◺       ◹      ◺      │   each ◹/◺ = board at a
        top │        ◺      every height &  │        different pose
            │   ◺       depth, tilted        │        (position + angle)
     bottom │      ◹        ◺        ◹       │
            └───────────────────────────────┘
```

---

## Part 3: The Session Folder Format

`sleap-anipose` is organized around a **session** folder. Each camera/view gets a
subfolder containing a `calibration_images/` subfolder. `calibrate()` writes
`calibration.toml` at the **session root**.

### Target structure

```
session/                              <- the "session" you pass to calibrate()
├── calibration.toml                  <- OUTPUT (what this tutorial produces)
├── calibration_metadata.h5           <- OUTPUT (detections + reprojections)
├── reprojection_histogram.png        <- OUTPUT (quality plot)
├── CAM1/
│   └── calibration_images/
│       └── session-CAM1-calibration.mp4    <- the calibration VIDEO for this view
├── CAM2/
│   └── calibration_images/
│       └── session-CAM2-calibration.mp4
├── ...
└── CAM6/
    └── calibration_images/
        └── session-CAM6-calibration.mp4
```

### How `calibrate()` actually finds the video

Straight from the installed `sleap_anipose/calibration.py`:

```python
calib_video = list(cam.glob("*/*calibration.mp4"))
if not calib_video:
    calib_videos.append([make_calibration_videos(cam.as_posix())])   # builds one from *.jpg
else:
    calib_videos.append([calib_video[0].as_posix()])                 # uses yours
```

So for each camera folder it looks **one directory deep for a file whose name ends in
`calibration.mp4`**. If it finds one, it uses it. If not, it falls back to stitching
`calibration_images/*.jpg` into a video for you.

> Earlier versions of this tutorial said `calibrate()` globs `*/*.MOV`. That is wrong —
> it is `*/*calibration.mp4`. A `.MOV` would be ignored and silently replaced by a
> JPEG-stitched video.

This tutorial writes the video itself, for two reasons.

### Reason 1: frame index is the ONLY link between cameras

Bundle adjustment relates two cameras through frames where **both** saw the board. In
`aniposelib`, that pairing is done purely by **frame number**:

```python
# aniposelib/boards.py -- merge_rows()
for cname, rows in zip(cam_names, all_rows):
    for r in rows:
        num = r['framenum']        # (video_index, frame_index)
        rows_dict[cname][num] = r  # grouped across cameras by this key alone
```

There is no timestamp and no content matching. **Frame `k` of CAM1's calibration video
is assumed to be the same instant as frame `k` of CAM2's.**

That makes the obvious-looking approach actively wrong: if each camera independently
keeps only *its own* good frames and those get renumbered `0..N`, then CAM1 frame 7 and
CAM2 frame 7 are unrelated moments, and the solver is fed board poses that never
co-occurred. Intrinsics survive (each camera only needs its own views), but the
**extrinsics — the entire point of multi-camera calibration — are garbage.**

So we choose **one shared list of source frame indices** and write that same list for
every camera. Frames where a given camera did not see the board are still written, as
placeholders, to keep the indices aligned. `aniposelib` simply finds no board there,
which is correct and harmless.

### Reason 2: the JPEG round-trip resizes your image

`make_calibration_videos()` calls `imageio.get_writer(fname, fps=30)` without
`macro_block_size`, so imageio-ffmpeg **pads the frame up to a multiple of 16**. A
1920x1080 recording comes back as **1920x1088**. `calibrate()` then reads the camera
resolution from that video, so your intrinsics are solved for an image 8 px taller than
the one your pose data actually came from — biasing the principal point `cy`.

Writing the video ourselves with `macro_block_size=1` keeps the resolution exact.

### Starting point

One raw calibration video per camera, all recording the same board motion at the same
time. Point `RAW_CALIB_VIDEOS` at them; the dict key becomes the camera folder name and
the camera's `name` inside `calibration.toml`.


In [ ]:
# ============================================================
# STEP 3: Point at your raw calibration videos (one per camera)
# ============================================================
from pathlib import Path

# ── EDIT: view/camera name -> its raw calibration video ──────
# These keys become the folder names and the camera names in calibration.toml.
RAW_CALIB_VIDEOS = {
    "CAM1": "/path/to/calibration/CAM1_calib.mp4",
    "CAM2": "/path/to/calibration/CAM2_calib.mp4",
    "CAM3": "/path/to/calibration/CAM3_calib.mp4",
    "CAM4": "/path/to/calibration/CAM4_calib.mp4",
    "CAM5": "/path/to/calibration/CAM5_calib.mp4",
    "CAM6": "/path/to/calibration/CAM6_calib.mp4",
}

# ── Frame alignment ──────────────────────────────────────────
# Frame index is the only thing linking cameras (see Part 3), so the videos must be
# frame-aligned. If a camera starts N frames late, put N here and it will be shifted
# into the common timeline. Leave everything 0 if your rig hardware-syncs.
FRAME_OFFSET = {view: 0 for view in RAW_CALIB_VIDEOS}
# e.g. FRAME_OFFSET["CAM5"] = 12   # CAM5 started 12 frames after the others

# ── Frame selection ──────────────────────────────────────────
SCAN_STRIDE = 5        # examine every Nth frame when looking for the board
MIN_MARKER_FRAC = 0.25 # a view "sees" the board if >= this fraction of markers detected
MIN_CAMS = 2           # keep a frame if at least this many views see the board.
                       # 2 is the minimum for triangulation; 3+ gives a stronger solve.
MAX_FRAMES = 250       # cap on kept frames (plenty for calibration)

# The session folder we will build and then calibrate.
SESSION = Path(PROJECT_DIR) / "session"
SESSION.mkdir(parents=True, exist_ok=True)

print(f"Session folder: {SESSION.resolve()}\n")
print(f"{'view':6s}  {'status':8s} {'frames':>8} {'resolution':>12}  video")
missing = []
VIDEO_INFO = {}
for view, vid in RAW_CALIB_VIDEOS.items():
    p = Path(vid)
    if not p.exists():
        missing.append(view)
        print(f"{view:6s}  {'MISSING':8s} {'-':>8} {'-':>12}  {vid}")
        continue
    cap = cv2.VideoCapture(str(p))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()
    VIDEO_INFO[view] = {"n_frames": n, "size": (w, h), "fps": fps}
    print(f"{view:6s}  {'OK':8s} {n:>8} {f'{w}x{h}':>12}  {p.name}")

if missing:
    print(f"\n(!) Update the paths above for: {', '.join(missing)}")
elif VIDEO_INFO:
    counts = {v: i["n_frames"] for v, i in VIDEO_INFO.items()}
    spread = max(counts.values()) - min(counts.values())
    print(f"\nFrame counts differ by {spread} frame(s) across cameras.")
    if spread > 5 and not any(FRAME_OFFSET.values()):
        print("(!) That is a lot. If the clips do not start at the same instant, set")
        print("    FRAME_OFFSET above -- otherwise the extrinsics will be solved from")
        print("    board poses that never co-occurred. See Part 3.")


---

## Part 4: Build Synchronized Calibration Videos

We now turn the raw videos into exactly what `calibrate()` wants:
`session/<view>/calibration_images/<session>-<view>-calibration.mp4`.

### Two passes

**Pass 1 — scan.** Walk every camera's video (every `SCAN_STRIDE`th frame) and record how
many ArUco markers are visible, indexed in the **common timeline** (source frame minus
that camera's `FRAME_OFFSET`).

**Pass 2 — write.** Choose the frames where **at least `MIN_CAMS` cameras** see the
board, subsample evenly to `MAX_FRAMES`, then write that *same index list* to every
camera's video.

```
common timeline:   0    5   10   15   20   25   30   35   40  ...
CAM1 sees board:   .    Y    Y    Y    .    .    Y    Y    Y
CAM2 sees board:   .    Y    Y    .    .    Y    Y    Y    .
CAM3 sees board:   .    .    Y    Y    .    Y    Y    .    .
                        ^    ^    ^         ^    ^    ^
                        └────┴────┴─────────┴────┴────┘
                        kept (>= 2 cameras see it), SAME list for every camera
```

Every output video ends up with the **same frame count**, and frame `k` means the same
instant in all of them. That is the property the whole solve rests on.

### Why keep frames a camera cannot see?

Because dropping them per-camera is exactly what breaks the index alignment. A camera
that missed the board at common frame 15 still gets frame 15 written; `aniposelib`
detects nothing there and moves on. Costless, and it keeps `merge_rows` honest.

### Notes

- Videos are written with `macro_block_size=1`, so **resolution is preserved exactly**
  (see Part 3, reason 2).
- Cameras may have different resolutions from each other — that is fine, each is solved
  with its own intrinsics.
- The detector below handles both `cv2.aruco` APIs; the module was reorganized between
  OpenCV 4.6 and 4.7.
- Set `ALSO_WRITE_JPGS = True` if you want the kept frames on disk to look at. They are
  written to a `qc_frames/` folder that `calibrate()` ignores — **not** to
  `calibration_images/`, where stray `.jpg` files would invite the resizing fallback.


In [ ]:
# ============================================================
# STEP 4a: ChArUco detector (handles old & new cv2.aruco APIs)
# ============================================================
import cv2
from cv2 import aruco

_ARUCO_DICTS = {
    (4, 50): aruco.DICT_4X4_50,   (4, 100): aruco.DICT_4X4_100,
    (4, 250): aruco.DICT_4X4_250, (4, 1000): aruco.DICT_4X4_1000,
    (5, 50): aruco.DICT_5X5_50,   (5, 100): aruco.DICT_5X5_100,
    (5, 250): aruco.DICT_5X5_250, (5, 1000): aruco.DICT_5X5_1000,
    (6, 50): aruco.DICT_6X6_50,   (6, 100): aruco.DICT_6X6_100,
    (6, 250): aruco.DICT_6X6_250, (6, 1000): aruco.DICT_6X6_1000,
    (7, 50): aruco.DICT_7X7_50,   (7, 100): aruco.DICT_7X7_100,
    (7, 250): aruco.DICT_7X7_250, (7, 1000): aruco.DICT_7X7_1000,
}

def make_detector(board=BOARD):
    'Return (detect_fn, n_total_markers). detect_fn(gray) -> n_markers_seen.'
    dict_id = _ARUCO_DICTS[(board["marker_bits"], board["dict_size"])]
    aruco_dict = aruco.getPredefinedDictionary(dict_id)

    # New API (OpenCV >= 4.7): ArucoDetector object
    if hasattr(aruco, "ArucoDetector"):
        params = aruco.DetectorParameters()
        det = aruco.ArucoDetector(aruco_dict, params)
        def detect(gray):
            corners, ids, _ = det.detectMarkers(gray)
            return 0 if ids is None else len(ids)
    # Old API (OpenCV <= 4.6): free functions
    else:
        params = aruco.DetectorParameters_create()
        def detect(gray):
            corners, ids, _ = aruco.detectMarkers(gray, aruco_dict, parameters=params)
            return 0 if ids is None else len(ids)

    # A ChArUco board of (bx, by) has floor(bx*by/2) markers.
    n_markers = (board["board_x"] * board["board_y"]) // 2
    return detect, n_markers

_detect_markers, N_MARKERS = make_detector()
print(f"Detector ready. Full board shows up to {N_MARKERS} ArUco markers.")

In [ ]:
# ============================================================
# STEP 4b: Scan all cameras, then write synchronized videos
# ============================================================
import shutil

import imageio

ALSO_WRITE_JPGS = False     # optional QC copies, written outside calibration_images/

min_markers = max(4, int(MIN_MARKER_FRAC * N_MARKERS))
views = [v for v in RAW_CALIB_VIDEOS if v in VIDEO_INFO]
print(f"A view 'sees' the board with >= {min_markers} of {N_MARKERS} markers.")
print(f"Keeping frames seen by >= {MIN_CAMS} of {len(views)} cameras, "
      f"max {MAX_FRAMES}, scan stride {SCAN_STRIDE}.\n")


# ── Pass 1: where is the board, per camera, in the common timeline ──
def scan_camera(view):
    """Return {common_frame_index: n_markers} for frames we examined."""
    cap = cv2.VideoCapture(str(RAW_CALIB_VIDEOS[view]))
    off = FRAME_OFFSET.get(view, 0)
    seen, idx = {}, 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        common = idx - off
        # Only examine frames that land on the shared stride grid, so every camera
        # examines the SAME common-timeline indices.
        if common >= 0 and common % SCAN_STRIDE == 0:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            seen[common] = _detect_markers(gray)
        idx += 1
    cap.release()
    return seen, idx


print("Pass 1 - scanning for the board:")
scans, n_src = {}, {}
for view in views:
    scans[view], n_src[view] = scan_camera(view)
    n_good = sum(1 for m in scans[view].values() if m >= min_markers)
    print(f"  {view:6s} examined {len(scans[view]):5d} frames, "
          f"board visible in {n_good:5d}")

# ── Choose ONE shared frame list ──────────────────────────────
all_common = sorted(set().union(*(set(s) for s in scans.values())))
votes = {f: sum(1 for v in views if scans[v].get(f, 0) >= min_markers)
         for f in all_common}
candidates = [f for f in all_common if votes[f] >= MIN_CAMS]

# Subsample evenly rather than taking the first N, so we keep coverage of the whole
# recording (and therefore the whole capture volume) instead of just the beginning.
if len(candidates) > MAX_FRAMES:
    sel = np.linspace(0, len(candidates) - 1, MAX_FRAMES).round().astype(int)
    KEEP_FRAMES = [candidates[k] for k in sorted(set(sel))]
else:
    KEEP_FRAMES = candidates

print(f"\n  {len(candidates)} frames seen by >= {MIN_CAMS} cameras "
      f"-> keeping {len(KEEP_FRAMES)}")
if KEEP_FRAMES:
    hist = {}
    for f in KEEP_FRAMES:
        hist[votes[f]] = hist.get(votes[f], 0) + 1
    print("  kept frames by number of cameras seeing the board:")
    for k in sorted(hist, reverse=True):
        print(f"    {k} cameras: {hist[k]:5d} frames")
    print(f"  span: common frame {KEEP_FRAMES[0]} .. {KEEP_FRAMES[-1]}")

if not KEEP_FRAMES:
    raise RuntimeError(
        "No frame was seen by enough cameras. Try lowering MIN_CAMS or "
        "MIN_MARKER_FRAC, reducing SCAN_STRIDE, or check FRAME_OFFSET -- if the clips "
        "are misaligned, no common frame will ever line up."
    )


# ── Pass 2: write the same frame list for every camera ────────
def write_view_video(view, keep):
    """Write session/<view>/calibration_images/<session>-<view>-calibration.mp4."""
    out_dir = SESSION / view / "calibration_images"
    if out_dir.exists():
        shutil.rmtree(out_dir)              # keep re-runs reproducible
    out_dir.mkdir(parents=True)

    # Name must end in "calibration.mp4" for calibrate()'s glob to find it.
    fname = out_dir / f"{SESSION.name}-{view}-calibration.mp4"
    fps = VIDEO_INFO[view]["fps"] or 30
    # macro_block_size=1 -> do NOT pad the frame up to a multiple of 16.
    writer = imageio.get_writer(str(fname), fps=fps, macro_block_size=1, quality=9)

    qc_dir = SESSION / view / "qc_frames"
    if ALSO_WRITE_JPGS:
        qc_dir.mkdir(parents=True, exist_ok=True)

    off = FRAME_OFFSET.get(view, 0)
    want = {f + off for f in keep}          # source indices for this camera
    order = {f + off: k for k, f in enumerate(keep)}
    grabbed = {}

    cap = cv2.VideoCapture(str(RAW_CALIB_VIDEOS[view]))
    idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if idx in want:
            grabbed[order[idx]] = frame
        idx += 1
    cap.release()

    # Write in common-timeline order, substituting black for frames this camera does
    # not have (e.g. an offset running past the end) so indices stay aligned.
    w, h = VIDEO_INFO[view]["size"]
    blank = np.zeros((h, w, 3), np.uint8)
    n_missing = 0
    for k in range(len(keep)):
        frame = grabbed.get(k)
        if frame is None:
            frame = blank
            n_missing += 1
        writer.append_data(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        if ALSO_WRITE_JPGS and grabbed.get(k) is not None:
            cv2.imwrite(str(qc_dir / f"{view}_common{keep[k]:06d}.jpg"), frame)
    writer.close()
    return fname, n_missing


print("\nPass 2 - writing synchronized videos:")
CALIB_VIDEOS = {}
for view in views:
    fname, n_missing = write_view_video(view, KEEP_FRAMES)
    note = f"  ({n_missing} blank placeholders)" if n_missing else ""
    print(f"  {view:6s} {len(KEEP_FRAMES):5d} frames -> "
          f"{fname.relative_to(SESSION)}{note}")
    CALIB_VIDEOS[view] = fname

print(f"\nDone. Every camera has {len(KEEP_FRAMES)} frames, index-aligned.")


In [ ]:
# ============================================================
# STEP 4c: Verify the videos before spending minutes on calibration
# ============================================================
# Three things must hold, and all three are silent failures if they don't:
#   1. calibrate()'s glob actually finds each video
#   2. every video has the SAME frame count  (cross-camera index alignment)
#   3. resolution is unchanged from the source (no macro-block padding)

print(f"{'view':6s} {'found by glob':>14} {'frames':>8} {'source':>12} "
      f"{'written':>12}  {'size ok':>8}")
problems = []
for view in views:
    cam_dir = SESSION / view
    globbed = list(cam_dir.glob("*/*calibration.mp4"))       # calibrate()'s exact glob

    cap = cv2.VideoCapture(str(CALIB_VIDEOS[view]))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    src_w, src_h = VIDEO_INFO[view]["size"]
    size_ok = (w, h) == (src_w, src_h)
    print(f"{view:6s} {len(globbed):>14} {n:>8} {f'{src_w}x{src_h}':>12} "
          f"{f'{w}x{h}':>12}  {'yes' if size_ok else 'NO':>8}")

    if len(globbed) != 1:
        problems.append(f"{view}: glob found {len(globbed)} videos, expected 1")
    if n != len(KEEP_FRAMES):
        problems.append(f"{view}: {n} frames written, expected {len(KEEP_FRAMES)}")
    if not size_ok:
        problems.append(f"{view}: resolution changed {src_w}x{src_h} -> {w}x{h}")
    if list((SESSION / view / "calibration_images").glob("*.jpg")):
        problems.append(f"{view}: stray .jpg in calibration_images/ "
                        f"(calibrate may rebuild the video from them)")

if problems:
    print("\nPROBLEMS:")
    for p in problems:
        print(f"  ! {p}")
else:
    print(f"\nAll {len(views)} videos OK: found by glob, "
          f"{len(KEEP_FRAMES)} frames each, resolution preserved.")

# ── Look at the same common frame in every camera ─────────────
# This is the visual version of the alignment claim: the board should be in the same
# real-world pose in every panel (seen from different angles).
CHECK_K = min(len(KEEP_FRAMES) // 2, len(KEEP_FRAMES) - 1)

dict_id = _ARUCO_DICTS[(BOARD["marker_bits"], BOARD["dict_size"])]
aruco_dict = aruco.getPredefinedDictionary(dict_id)
if hasattr(aruco, "ArucoDetector"):
    _det = aruco.ArucoDetector(aruco_dict, aruco.DetectorParameters())
    _detect_full = lambda g: _det.detectMarkers(g)
else:
    _detect_full = lambda g: aruco.detectMarkers(
        g, aruco_dict, parameters=aruco.DetectorParameters_create())

ncol = min(3, len(views))
nrow = int(np.ceil(len(views) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(7 * ncol, 4.5 * nrow))
axes = np.atleast_1d(axes).ravel()

for ax, view in zip(axes, views):
    cap = cv2.VideoCapture(str(CALIB_VIDEOS[view]))
    cap.set(cv2.CAP_PROP_POS_FRAMES, CHECK_K)
    ok, img = cap.read()
    cap.release()
    if not ok:
        ax.text(0.5, 0.5, f"{view}: could not read frame", ha="center", va="center")
        ax.axis("off")
        continue
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    corners, ids, _ = _detect_full(gray)
    aruco.drawDetectedMarkers(img, corners, ids)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{view}: {0 if ids is None else len(ids)} markers", fontsize=11)
    ax.axis("off")
for ax in axes[len(views):]:
    ax.axis("off")

fig.suptitle(f"Video frame {CHECK_K} (common source frame {KEEP_FRAMES[CHECK_K]}) "
             f"in every camera -- the board should be in the SAME pose, seen from "
             f"different angles", fontsize=12)
plt.tight_layout()
plt.show()
print("If the board is in visibly different positions across these panels, the clips "
      "are not frame-aligned -- fix FRAME_OFFSET in STEP 3 before calibrating.")


---

## Part 5: Run Calibration → `calibration.toml`

Everything is now in the format `slap.calibrate()` expects. A single call:

1. Discovers the camera folders (each subfolder of the session is one view).
2. Finds the `*calibration.mp4` we wrote in Part 4 and uses it as-is — no JPEG
   stitching, no resizing.
3. Detects ChArUco corners across all views and runs **iterative bundle adjustment**,
   pairing cameras by frame index.
4. Writes `calibration.toml` (the deliverable) plus optional QC outputs.

### Key arguments

| Argument | What it does |
|---|---|
| `session` | Path to the session folder (its subfolders are the views) |
| `board` | Board spec — a dict, a `CharucoBoard`, or the `board.toml` path |
| `calib_fname` | Where to write `calibration.toml` (**the main output**) |
| `metadata_fname` | `.h5` of detections + triangulations + reprojections (QC) |
| `histogram_path` | `.png` reprojection-error histogram (QC) |
| `reproj_path` | Folder to write detected-vs-reprojected overlay images (QC) |
| `excluded_views` | View **names** to leave out (e.g. a broken camera) |

> Calibration is CPU-heavy and can take a few minutes for 6 cameras x hundreds of
> frames. That's normal.
>
> `aniposelib` examines every 20th frame by default, but after each successful board
> detection it densely examines the next ~10. Since Part 4 kept only frames where the
> board was actually visible, that streak behaviour means effectively all of them get
> used.

In [ ]:
# ============================================================
# STEP 5: Calibrate the session
# ============================================================
calib_toml   = SESSION / "calibration.toml"
metadata_h5  = SESSION / "calibration_metadata.h5"
histogram_png = SESSION / "reprojection_histogram.png"

cgroup, metadata = slap.calibrate(
    session=str(SESSION),
    board=BOARD,                       # dict is fine; could also pass board_toml
    excluded_views=(),                 # e.g. ("CAM5",) to drop a bad camera
    calib_fname=str(calib_toml),       # <-- produces calibration.toml
    metadata_fname=str(metadata_h5),
    histogram_path=str(histogram_png),
    reproj_path=str(SESSION),          # writes reprojection-*.png into each view
)

frames, detections, triangulations, reprojections = metadata
print(f"\nDone. Calibrated {len(cgroup.get_names())} cameras: {cgroup.get_names()}")
print(f"Common board frames used across all views: {len(frames)}")
print(f"calibration.toml -> {calib_toml.resolve()}")

---

## Part 6: Reading `calibration.toml`

The output is a plain TOML file with one `[cam_...]` block per camera. This is exactly
the file **Tutorial 3** loads to triangulate.

```toml
[cam_0]
name = "CAM1"
size = [1920, 1080]                                 # image resolution (px)
matrix = [[fx, 0, cx], [0, fy, cy], [0, 0, 1]]      # intrinsics
distortions = [k1, k2, p1, p2, k3]                  # lens distortion
rotation = [rx, ry, rz]                             # extrinsics (Rodrigues rvec)
translation = [tx, ty, tz]                          # extrinsics (in board units)
```

- **`matrix`** — intrinsic matrix. `fx, fy` focal lengths (px); `cx, cy` principal point.
- **`distortions`** — radial (`k1,k2,k3`) + tangential (`p1,p2`) lens distortion.
- **`rotation` / `translation`** — where the camera sits in the shared world frame.
  `translation` is in **your board units** (mm if you used mm), which is why the board
  measurement sets your 3D scale.

In [ ]:
# ============================================================
# STEP 6: Print and parse calibration.toml
# ============================================================
import toml

print(calib_toml.read_text()[:2000])
print("..." if calib_toml.stat().st_size > 2000 else "")

calib = toml.load(calib_toml)
print("\nParsed per-camera summary:")
for key, cam in calib.items():
    if key == "metadata":
        continue
    mat = np.array(cam["matrix"])
    print(f"  {cam.get('name', key):6s}  size={cam['size']}  "
          f"fx={mat[0,0]:7.1f}  fy={mat[1,1]:7.1f}  "
          f"cx={mat[0,2]:6.1f}  cy={mat[1,2]:6.1f}  "
          f"|t|={np.linalg.norm(cam['translation']):.1f}")

---

## Part 7: Judging Calibration Quality

**Never trust a calibration you haven't checked.** The single best metric is
**reprojection error**: triangulate the detected board corners back to 3D, project them
into every camera, and measure the pixel distance from the original detections.

### Rules of thumb

| Mean reprojection error | Verdict |
|---|---|
| **< 1 px** | Excellent |
| **1–3 px** | Good — fine for most 3D pose work |
| **3–5 px** | Marginal — usable but consider re-recording |
| **> 5 px** | Poor — re-record with better board coverage, or exclude a bad view |

If one camera is dragging the error up, re-run with that view in `excluded_views`, or
re-record calibration video for it (usually it never shared enough board views with the
others).

In [ ]:
# ============================================================
# STEP 7a: Reprojection-error histogram + per-camera breakdown
# ============================================================
# detections / reprojections: (n_cams, n_frames, n_corners, 2)
err = np.linalg.norm(detections - reprojections, axis=-1)   # (n_cams, n_frames, n_corners)
per_cam = np.nanmean(err.reshape(err.shape[0], -1), axis=1)

print("Per-camera mean reprojection error (px):")
for name, e in zip(cgroup.get_names(), per_cam):
    flag = "  <-- check this view" if e > 5 else ""
    print(f"  {name:6s}  {e:5.2f} px{flag}")
print(f"\nOverall mean: {np.nanmean(err):.2f} px   median: {np.nanmedian(err):.2f} px")

plt.figure(figsize=(8, 5))
plt.hist(err.ravel()[~np.isnan(err.ravel())], bins=np.linspace(0, 15, 60), density=True)
plt.axvline(np.nanmean(err), color="r", ls="--", label=f"mean {np.nanmean(err):.2f}px")
plt.xlabel("Reprojection error (px)")
plt.ylabel("PDF")
plt.title("Reprojection error across all views")
plt.legend()
plt.show()

# The saved histogram from calibrate():
if histogram_png.exists():
    print(f"\nSaved histogram -> {histogram_png}")

In [ ]:
# ============================================================
# STEP 7b: Look at a saved detection-vs-reprojection overlay
# ============================================================
# calibrate(reproj_path=...) drops reprojection-*.png into each view folder.
# Red '+' = detected corners, green 'x' = reprojected corners. They should overlap.
overlays = sorted(SESSION.glob("*/reprojection-*.png"))
if overlays:
    show = overlays[:min(3, len(overlays))]
    fig, axes = plt.subplots(1, len(show), figsize=(6 * len(show), 6))
    axes = np.atleast_1d(axes)
    for ax, p in zip(axes, show):
        ax.imshow(plt.imread(p))
        ax.set_title(f"{p.parent.name}/{p.name}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No reprojection overlays found (pass reproj_path= to calibrate to generate them).")

---

## Summary & Handoff to Tutorial 3

You went from **raw calibration videos → `calibration.toml`**:

```
raw per-camera calibration videos  (frame-aligned)
        |  Part 4 pass 1: scan every camera for the board
        |  Part 4 pass 2: write ONE shared frame list per camera
        v
session/<CAM>/calibration_images/<session>-<CAM>-calibration.mp4
        |         same frame count, index-aligned, resolution preserved
        |  Part 5: slap.calibrate(...)
        v
session/calibration.toml                     <- the deliverable
   + calibration_metadata.h5, reprojection_histogram.png   (QC)
```

### Using it in the pipeline

Drop `calibration.toml` at the root of your **triangulation** session (the folder with
`cam1/ … cam6/` pose `.analysis.h5` files in Tutorial 3) and point the triangulation
step at it:

```python
import sleap_anipose as slap
slap.triangulate(
    p2d="/path/to/triangulation_session",
    calib="/path/to/triangulation_session/calibration.toml",   # <-- from THIS tutorial
    fname="points3d.h5",
)
```

Because `translation` in `calibration.toml` is in the **units of your board's
`square_length`**, your 3D coordinates in Tutorial 3 come out in those same units.

### Checklist for a good calibration

- [ ] Board printed at 100% scale, flat & rigid; `square_length`/`marker_length` measured
- [ ] Calibration clips are **frame-aligned** across cameras (`FRAME_OFFSET` set if not)
- [ ] STEP 4c reports the **same frame count** for every camera and **unchanged resolution**
- [ ] STEP 4c's grid shows the board in the **same pose** in every camera
- [ ] **100–300** kept frames, most seen by **3+** cameras
- [ ] Board was moved through the **whole volume**, tilted at many angles
- [ ] Mean reprojection error **< 3 px**; no single camera is an outlier

> If your reprojection error sits stubbornly in the 5-20 px range, suspect frame
> alignment before you suspect the board or the optics. Per-camera frame selection
> (renumbering each camera's good frames to `0..N` independently) produces exactly that
> signature: plausible intrinsics, quietly wrong extrinsics.
- [ ] `calibration.toml` copied to the triangulation session for Tutorial 3

### Troubleshooting

| Symptom | Likely fix |
|---|---|
| `cv2.aruco` missing | `pip install opencv-contrib-python` (remove plain `opencv-python`) |
| Few/no frames kept | Lower `MIN_MARKER_FRAC` / `FRAME_STRIDE`; check lighting & focus |
| High error on one camera | Add it to `excluded_views`, or re-record that view |
| High error everywhere | Board bent, wrong `square_length`, or too little volume coverage |
| Wrong 3D scale later | `square_length` didn't match the real printed board |